# 🧠 Notebook 1: OO Analysis & Design + SOLID

This notebook is **runnable**. Every SOLID principle is demonstrated with a tiny bad-version and good-version you can execute and compare.

## 🛠️ Setup

```bash
cd 07-object-oriented-design/oo-analysis-and-design
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 1. From requirements → design (the process)

When someone says *'build a parking lot system'*, don't jump to code. The OO design process is roughly:

1. **Gather requirements** — functional (what it does) *and* non-functional (how fast, how many users).
2. **Find the nouns** → candidate **classes**. (ParkingLot, Vehicle, Ticket, Slot…)
3. **Find the verbs** → candidate **methods**. (park, leave, price…)
4. **Draw relationships** (has-a, is-a, uses).
5. **Sanity-check with SOLID**.
6. **Write code**, iterate.

The point is not bureaucracy — it's avoiding the #1 junior mistake: *coding before understanding*.

### 🔎 Tiny noun/verb extractor

Let's do the exercise on a real requirement. We'll use a very simple heuristic (words ending in common noun/verb patterns). Real analysts use judgment, not regex — this is just to make the process tangible.

In [ ]:
import re

REQUIREMENT = '''
The parking lot has multiple floors. Each floor has parking slots of different sizes.
A vehicle enters, takes a ticket, parks in a slot, and later pays and leaves.
The system tracks capacity and computes the price based on duration.
'''

STOPWORDS = {'the','a','an','of','and','or','in','on','at','to','it','is','are',
             'has','have','based','different','multiple','later','each'}

words = re.findall(r'[A-Za-z]+', REQUIREMENT.lower())
words = [w for w in words if w not in STOPWORDS]

# naive heuristic: verbs often end in -s/-es/-ks/-ys after a subject; nouns are the rest
LIKELY_VERBS = {'enters','takes','parks','pays','leaves','tracks','computes'}
nouns = sorted({w for w in words if w not in LIKELY_VERBS})
verbs = sorted({w for w in words if w in LIKELY_VERBS})

print('Candidate classes (nouns):', nouns)
print('Candidate methods (verbs):', verbs)

From that list a designer would pick the *important* nouns (ParkingLot, Floor, Slot, Vehicle, Ticket) and drop noise (system, capacity, duration become attributes, not classes). The verbs become methods on the right class: `lot.park(vehicle)`, `ticket.price()`, etc.

## 2. SOLID — 5 principles that survive time

We'll run each principle as **bad → good**. Don't memorize the acronym; learn the *smell* each principle prevents.

### S — Single Responsibility Principle (SRP)
> A class should have **one reason to change**.

Smell: a class that changes whenever *anything* changes (DB schema, email template, tax law…).

In [ ]:
# ❌ BAD: User knows about storage AND email AND formatting
class BadUser:
    def __init__(self, name, email):
        self.name, self.email = name, email
    def save_to_db(self):
        print(f'  INSERT INTO users VALUES ({self.name!r})')
    def send_welcome_email(self):
        print(f'  SMTP → {self.email}: welcome {self.name}!')
    def to_json(self):
        return f'{{"name": "{self.name}"}}'

BadUser('Alice', 'a@x.com').save_to_db()
BadUser('Alice', 'a@x.com').send_welcome_email()

In [ ]:
# ✅ GOOD: one class, one reason to change
from dataclasses import dataclass

@dataclass
class User:
    name: str
    email: str

class UserRepository:
    def save(self, user: User):
        print(f'  INSERT INTO users VALUES ({user.name!r})')

class WelcomeEmailer:
    def send(self, user: User):
        print(f'  SMTP → {user.email}: welcome {user.name}!')

u = User('Alice', 'a@x.com')
UserRepository().save(u)
WelcomeEmailer().send(u)

### O — Open/Closed Principle (OCP)
> Open for **extension**, closed for **modification**.

Smell: every new feature means editing the same `if/elif` ladder.

In [ ]:
# ❌ BAD: adding a new shape means editing this function
def bad_area(shape):
    if shape['kind'] == 'circle':
        return 3.14159 * shape['r'] ** 2
    elif shape['kind'] == 'square':
        return shape['side'] ** 2
    # add triangle? edit here. add ellipse? edit here. forever.
    raise ValueError('unknown shape')

print('circle:', bad_area({'kind': 'circle', 'r': 2}))
print('square:', bad_area({'kind': 'square', 'side': 3}))

In [ ]:
# ✅ GOOD: add a new shape by writing a new class — no edits to old code
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return math.pi * self.r ** 2

class Square(Shape):
    def __init__(self, side): self.side = side
    def area(self): return self.side ** 2

# New requirement → new file, zero edits above
class Triangle(Shape):
    def __init__(self, base, height): self.base, self.height = base, height
    def area(self): return 0.5 * self.base * self.height

for s in [Circle(2), Square(3), Triangle(4, 5)]:
    print(type(s).__name__, '→', round(s.area(), 2))

### L — Liskov Substitution Principle (LSP)
> A subclass must be usable **wherever** its parent is used, without surprises.

Smell: code that says `if isinstance(x, Penguin): skip fly()`. That's a hierarchy bug.

In [ ]:
# ❌ BAD: Penguin IS-A Bird, but breaks the contract when substituted
class Bird:
    def fly(self): return 'flying'

class Penguin(Bird):
    def fly(self):
        raise NotImplementedError("penguins can't fly!")

def make_them_fly(birds):
    # This function assumed every Bird can fly. Penguin breaks that assumption.
    return [b.fly() for b in birds]

try:
    make_them_fly([Bird(), Penguin()])
except NotImplementedError as e:
    print('💥 LSP violated:', e)

In [ ]:
# ✅ GOOD: split the hierarchy by *capability*, not by biology
class Bird:                       # base: everything every bird has
    def __init__(self, name): self.name = name
    def eat(self): return f'{self.name} eats'

class FlyingBird(Bird):
    def fly(self): return f'{self.name} is flying'

class SwimmingBird(Bird):
    def swim(self): return f'{self.name} is swimming'

class Sparrow(FlyingBird): pass
class Penguin(SwimmingBird): pass

def make_them_fly(flyers: list[FlyingBird]):
    # No isinstance checks, no try/except: the TYPE says who belongs here.
    return [f.fly() for f in flyers]

print(make_them_fly([Sparrow('Jack'), Sparrow('Jill')]))

# The key LSP property: EVERY FlyingBird can stand in for FlyingBird...
assert all('flying' in r for r in make_them_fly([Sparrow('a')]))

# ...and Penguin simply isn't one, so it can never reach this function by accident.
assert not isinstance(Penguin('Pingu'), FlyingBird)
assert not hasattr(Penguin('Pingu'), 'fly'), 'a penguin should not even HAVE fly()'

# Both are still Birds where being a bird is all that matters -- we split the
# hierarchy, we did not shatter it.
assert all(isinstance(b, Bird) for b in [Sparrow('a'), Penguin('b')])
print(Penguin('Pingu').swim(), '|', Penguin('Pingu').eat())

# 💡 The BAD version needed a runtime `try/except NotImplementedError`.
#    The GOOD version needs no runtime check at all -- a type checker (mypy,
#    pyright) rejects `make_them_fly([Penguin(...)])` before the code ever runs.
#    "Push errors from runtime to compile time" is what LSP buys you.

### I — Interface Segregation Principle (ISP)
> Many small interfaces > one fat interface.

Smell: classes full of `raise NotImplementedError` because they inherit methods they don't need.

In [ ]:
# ❌ BAD: fat interface — SimplePrinter is forced to implement scan/fax it doesn't support
from abc import ABC, abstractmethod

class MultiFunctionDevice(ABC):
    @abstractmethod
    def print(self, doc): ...
    @abstractmethod
    def scan(self, doc): ...
    @abstractmethod
    def fax(self, doc): ...

class SimplePrinter(MultiFunctionDevice):
    def print(self, doc): print(f'printing {doc}')
    def scan(self, doc): raise NotImplementedError  # 🚩
    def fax(self, doc):  raise NotImplementedError  # 🚩

sp = SimplePrinter()
sp.print('report.pdf')
try:
    sp.scan('x')
except NotImplementedError:
    print('💥 SimplePrinter was forced to implement scan()')

In [ ]:
# ✅ GOOD: split into tiny, focused interfaces; mix only what you actually do
class Printer(ABC):
    @abstractmethod
    def print(self, doc): ...

class Scanner(ABC):
    @abstractmethod
    def scan(self, doc): ...

class Fax(ABC):
    @abstractmethod
    def fax(self, doc): ...

class SimplePrinter(Printer):
    def print(self, doc): print(f'printing {doc}')

class AllInOne(Printer, Scanner, Fax):
    def print(self, doc): print(f'printing {doc}')
    def scan(self, doc):  print(f'scanning {doc}')
    def fax(self, doc):   print(f'faxing {doc}')

SimplePrinter().print('invoice.pdf')
AllInOne().scan('contract.pdf')

### D — Dependency Inversion Principle (DIP)
> Depend on **abstractions**, not **concretions**.

Smell: `OrderService` does `import stripe` at the top of the file — now you can't test without hitting the real Stripe API.

In [ ]:
# ❌ BAD: OrderService is welded to Stripe. Hard to test, hard to swap.
class StripeClient:
    def charge(self, amount, token):
        print(f'  [REAL STRIPE] charged ${amount} with {token}')
        return 'ok'

class BadOrderService:
    def __init__(self):
        self.stripe = StripeClient()  # hardcoded!
    def checkout(self, amount, token):
        return self.stripe.charge(amount, token)

BadOrderService().checkout(42, 'tok_live_abc')

In [ ]:
# ✅ GOOD: depend on a PaymentGateway interface. Inject any implementation.
class PaymentGateway(ABC):
    @abstractmethod
    def charge(self, amount: float, token: str) -> str: ...

class StripeGateway(PaymentGateway):
    def charge(self, amount, token):
        print(f'  [stripe] charged ${amount}')
        return 'ok'

class FakeGateway(PaymentGateway):
    def __init__(self): self.calls = []
    def charge(self, amount, token):
        self.calls.append((amount, token))
        return 'ok'

class OrderService:
    def __init__(self, gateway: PaymentGateway):
        self.gateway = gateway
    def checkout(self, amount, token):
        return self.gateway.charge(amount, token)

# Production
OrderService(StripeGateway()).checkout(42, 'tok_live')

# Test — no network, no secrets
fake = FakeGateway()
assert OrderService(fake).checkout(10, 'tok_test') == 'ok'
assert fake.calls == [(10, 'tok_test')]
print('✅ unit test passed without calling real Stripe')

## 3. How to apply it in an interview / real project

**Don't optimize for 'following all of SOLID'.** Optimize for: *given likely future changes, is this code easy to extend?*

Heuristic checklist when reviewing a class:
- [ ] Does its name describe **one** concept? *(SRP)*
- [ ] Could I add a new variant without editing this file? *(OCP)*
- [ ] Can every subclass stand in for the parent? *(LSP)*
- [ ] Does this class implement methods it doesn't actually need? *(ISP)*
- [ ] Can I swap its dependencies for fakes in tests? *(DIP)*

If you can answer **yes** to all five, you're in great shape — even if you never uttered the word 'SOLID'.

## 4. Further reading

- *Clean Code* and *Clean Architecture* by Robert C. Martin (who popularized SOLID).
- *Refactoring* by Martin Fowler — when to apply these, and smells to look for.
- Next up in this repo: [`notebooks/02_refactor_to_solid.ipynb`](02_refactor_to_solid.ipynb) — a realistic god-class refactored step by step.

## 5. 🧪 Verify the principles

Prose about SOLID is easy to nod along to. These assertions check that the "good"
versions above really have the property each principle promises -- and they will
fail loudly if someone edits a cell and breaks it.

In [ ]:
def must_raise(exc, fn, *a, **kw):
    try:
        fn(*a, **kw)
    except exc:
        return True
    raise AssertionError(f'expected {exc.__name__}, nothing was raised')

# --- S: the data class no longer knows about storage or email -------------
assert not hasattr(User('x', 'y'), 'save_to_db')
assert not hasattr(User('x', 'y'), 'send_welcome_email')
# ...and each collaborator can be used on its own, with no User method involved.
assert callable(UserRepository().save) and callable(WelcomeEmailer().send)

# --- O: a brand-new shape works with code that predates it ----------------
class Ellipse(Shape):                       # written "after" the loop below existed
    def __init__(self, a, b): self.a, self.b = a, b
    def area(self): return math.pi * self.a * self.b

shapes = [Circle(1), Square(2), Triangle(3, 4), Ellipse(2, 3)]
assert [round(s.area(), 2) for s in shapes] == [3.14, 4, 6.0, 18.85]
# The caller above has zero branches on type -- that is the OCP payoff.
must_raise(TypeError, Shape)                # the abstraction itself is not instantiable

# --- L: every subtype is substitutable for its declared supertype ---------
assert all(isinstance(b, FlyingBird) and b.fly() for b in [Sparrow('s1'), Sparrow('s2')])
assert not isinstance(Penguin('p'), FlyingBird)

# --- I: a small implementation is not forced to stub methods it lacks -----
assert not hasattr(SimplePrinter(), 'scan'), 'ISP: no NotImplementedError stubs left'
assert hasattr(AllInOne(), 'scan') and hasattr(AllInOne(), 'fax')
assert issubclass(AllInOne, Printer) and issubclass(AllInOne, Scanner)
assert issubclass(SimplePrinter, Printer) and not issubclass(SimplePrinter, Scanner)

# --- D: the service works with a fake, with no network and no monkeypatching
fake = FakeGateway()
assert OrderService(fake).checkout(99, 'tok_fake') == 'ok'
assert fake.calls == [(99, 'tok_fake')]
# Swapping the implementation requires zero changes to OrderService:
assert OrderService(StripeGateway()).checkout(1, 'tok') == 'ok'
# And the abstraction refuses an incomplete implementation.
class Broken(PaymentGateway): pass
must_raise(TypeError, Broken)

print('\n✅ all five principles verified')